In [ ]:
import polars as pl

from collections import Counter

from rich.pretty import pprint
from common.utils.json_data import save_json

In [ ]:
ARTIST_MAP = {
    "Taylor Swift": "Taylor Swift",
    "The Doors": "The Doors",
    "Bjrk": "Björk",
    "David Bowie": "David Bowie",
    "The Smiths": "The Smiths",
    "Marika Hackman": "Marika Hackman",
    "The D": "The Dø",
    "The Growlers": "The Growlers",
    "Portishead": "Portishead",
    # "The Buttertones": "The Buttertones",
    "PJ Harvey": "PJ Harvey",
    "Massive Attack": "Massive Attack",
    "Gorillaz": "Gorillaz",
    "Ramones": "Ramones",
    "Gustavo Cerati": "Gustavo Cerati",
    "Arcade Fire": "Arcade Fire",
    "Charly Garca": "Charly García",
    # "Dua Lipa": "Dua Lipa",
    "The Beatles": "The Beatles",
    "Queen": "Queen",
    "Talking Heads": "Talking Heads",
    "Radiohead": "Radiohead",
    "Madonna": "Madonna",
    "The Rolling Stones": "The Rolling Stones",
    "Elvis Presley": "Elvis Presley",
    "Yeah Yeah Yeahs": "Yeah Yeah Yeahs",
    "Molly Nilsson": "Molly Nilsson",
    "Sumo": "Sumo",
    "John Maus": "John Maus",
    "The Sound": "The Sound",
    "The Clash": "The Clash",
    "Cranes": "Cranes",
    "Virus": "Virus",
    "Depeche Mode": "Depeche Mode",
    "Blonde Redhead": "Blonde Redhead",
    "MGMT": "MGMT",
    "The Police": "The Police",
    "Metronomy": "Metronomy",
    "Mac DeMarco": "Mac DeMarco",
    "Alexandra Savior": "Alexandra Savior",
    "Babasonicos": "Babasonicos",
    "The Cure": "The Cure",
    "AURORA": "AURORA",
    "Backstreet Boys": "Backstreet Boys",
    "Spice Girls": "Spice Girls",
    "Pescado Rabioso": "Pescado Rabioso",
}

In [ ]:
reader = pl.read_csv_batched(
    source="/resources/datasets/wasabi/wasabi_songs.csv",
    infer_schema_length=10_000,
    separator="\t",
    n_threads=4,
)

artist_values = set(ARTIST_MAP.values())
wasabi_songs = []

while True:
    batch = reader.next_batches(1)
    if batch is None:
        break

    for df in batch:
        data_items = df.to_dicts()
        for data_item in data_items:
            if data_item["artist"] in artist_values:
                wasabi_songs.append(data_item)

print(f"wasabi_songs => {len(wasabi_songs)}")

In [ ]:
diff = set(ARTIST_MAP.values()) - set(ws["artist"] for ws in wasabi_songs)
assert not diff, diff
pprint(wasabi_songs[0])

In [ ]:
Counter(ws["artist"] for ws in wasabi_songs)

In [ ]:
reader = pl.read_csv_batched(
    source="/resources/datasets/song_lyrics.csv",
    n_threads=4,
)

lyrics = []
while True:
    batch = reader.next_batches(1)
    if batch is None:
        break

    for df in batch:
        data_items = df.to_dicts()
        for data_item in data_items:
            if data_item["artist"] in ARTIST_MAP:
                lyrics.append(
                    data_item
                    | {
                        "artist": ARTIST_MAP[data_item["artist"]],
                    }
                )

print(f"lyrics => {len(lyrics)}")

In [ ]:
diff = set(ARTIST_MAP.values()) - set(li["artist"] for li in lyrics)
assert not diff, diff
pprint(lyrics[0])

In [ ]:
Counter(li["artist"] for li in lyrics)

In [ ]:
wasabi_songs_ = []
not_found_lyrics = []

lyrics_map = {f"{li['title']} | {li['artist']}": li for li in lyrics}
for ws in wasabi_songs:
    key = f"{ws['title']} | {ws['artist']}"
    lyrics_item = lyrics_map.get(key)
    if lyrics_item is None:
        not_found_lyrics.append(key)
        continue

    wasabi_songs_.append(ws | {"lyrics": lyrics_item["lyrics"]})


print(f"wasabi_songs: {len(wasabi_songs_)}")
print(f"not_found_lyrics: {len(not_found_lyrics)}")

In [ ]:
Counter(ws["artist"] for ws in wasabi_songs_)

In [ ]:
spotify_wasabi_songs = [
    ws for ws in wasabi_songs_ if ws["urlSpotify"] is not None
]
print(f"spotify_wasabi_songs: {len(spotify_wasabi_songs)}")

In [ ]:
Counter(ws["artist"] for ws in spotify_wasabi_songs)

In [ ]:
[sws["title"] for sws in spotify_wasabi_songs if sws["artist"] == "Madonna"]

In [ ]:
file_path = "/resources/documents/lyrics/lyrics.json"
save_json(obj=lyrics, file_path=file_path)